<a href="https://colab.research.google.com/github/jaysulk/GENERIC-FNO/blob/main/GENERIC_FNO_4_PDE_Experiment_in_2D.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
#!/usr/bin/env python3
"""
GENERIC-FNO Benchmark — 2D
==========================
Fourier Neural Operator with GENERIC thermodynamic structure, in 2D.

Architecture (unchanged from 1D, lifted to 2D):
  - E-net (FNO2d → scalar): learns energy functional E[u]
  - S-net (FNO2d → scalar): learns entropy functional S[u]
  - L(kx,ky): anti-Hermitian diagonal operator (reversible dynamics)
  - M(kx,ky): Hermitian PSD diagonal operator (dissipative dynamics)
  - Dynamics: du/dt = L·δE/δu + M·δS/δu
  - Hard projection: energy conservation + entropy non-decrease

Compares: FNO (vanilla), EP-FNO (energy penalty), GENERIC-FNO (ours)
Tests on: Heat, Wave, Burgers  (all 2D scalar fields)

Default resolution NX=128 (i.e. 128x128 grids). Adjust in run_benchmark().

Key insight: dynamics are CONSTRUCTED from E,S — no bypass possible.
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pickle
import time
import math
from collections import defaultdict


# ============================================================================
# Data Generation — 2D PDEs (spectral)
# ============================================================================

def _random_field_2d(nx, ny, max_mode, n_modes, amp_scale=0.5, device='cpu'):
    """Build a random smooth 2D field as a sum of sine modes."""
    x = torch.linspace(0, 2*math.pi, nx+1, device=device)[:-1]
    y = torch.linspace(0, 2*math.pi, ny+1, device=device)[:-1]
    X, Y = torch.meshgrid(x, y, indexing='ij')
    u = torch.zeros(nx, ny, device=device)
    for _ in range(n_modes):
        kx = torch.randint(1, max_mode, (1,)).item()
        ky = torch.randint(1, max_mode, (1,)).item()
        amp = torch.randn(1).item() * amp_scale
        phase = torch.rand(1).item() * 2 * math.pi
        u += amp * torch.sin(kx * X + ky * Y + phase)
    return u


def generate_heat_data_2d(n_samples=150, nx=128, nt=15, dt=0.005, nu=0.02, device='cpu'):
    """2D heat: du/dt = nu*(uxx+uyy). Purely dissipative. Exact in spectral space."""
    kx = torch.fft.fftfreq(nx, d=1.0/nx).to(device)
    ky = torch.fft.rfftfreq(nx, d=1.0/nx).to(device)
    KX, KY = torch.meshgrid(kx, ky, indexing='ij')
    k_sq = KX**2 + KY**2
    decay = torch.exp(-nu * k_sq * dt)

    data_in, data_out = [], []
    for _ in range(n_samples):
        n_modes = torch.randint(3, 7, (1,)).item()
        u0 = _random_field_2d(nx, nx, nx//8, n_modes, device=device)
        u_hat = torch.fft.rfft2(u0)
        traj = [u0.clone()]
        for t in range(nt):
            u_hat = u_hat * decay
            traj.append(torch.fft.irfft2(u_hat, s=(nx, nx)))
        traj = torch.stack(traj, dim=0)  # (nt+1, nx, nx)
        data_in.append(traj[:-1])
        data_out.append(traj[1:])
    return torch.stack(data_in), torch.stack(data_out), 'heat'


def generate_wave_data_2d(n_samples=150, nx=128, nt=15, dt=0.005, c=1.0, device='cpu'):
    """2D wave: u_tt = c²(uxx+uyy). Reversible. Track u-component, exact spectral rotation."""
    kx = torch.fft.fftfreq(nx, d=1.0/nx).to(device)
    ky = torch.fft.rfftfreq(nx, d=1.0/nx).to(device)
    KX, KY = torch.meshgrid(kx, ky, indexing='ij')
    kmag = torch.sqrt(KX**2 + KY**2)
    omega = c * kmag

    data_in, data_out = [], []
    for _ in range(n_samples):
        n_modes = torch.randint(3, 7, (1,)).item()
        u0 = _random_field_2d(nx, nx, nx//8, n_modes, device=device)
        v0 = _random_field_2d(nx, nx, nx//8, n_modes, amp_scale=0.3, device=device)
        u_hat = torch.fft.rfft2(u0)
        v_hat = torch.fft.rfft2(v0)
        traj = [u0.clone()]
        for t in range(nt):
            cos_w = torch.cos(omega * dt)
            sin_w = torch.sin(omega * dt)
            # rotate (u, v) preserving energy; guard omega=0 mode
            safe_omega = torch.where(omega > 1e-8, omega, torch.ones_like(omega))
            u_new = cos_w * u_hat + (sin_w / safe_omega) * v_hat
            v_new = -safe_omega * sin_w * u_hat + cos_w * v_hat
            u_hat, v_hat = u_new, v_new
            traj.append(torch.fft.irfft2(u_hat, s=(nx, nx)))
        traj = torch.stack(traj, dim=0)
        data_in.append(traj[:-1])
        data_out.append(traj[1:])
    return torch.stack(data_in), torch.stack(data_out), 'wave'


def generate_advection_data_2d(n_samples=150, nx=128, nt=15, dt=0.005, c=1.0,
                               max_mode=6, device='cpu'):
    """2D linear advection: u_t + c(u_x+u_y) = 0. Reversible, Markovian in u,
    conserves 0.5<u^2> exactly. Clean fully-observed reversible scalar test
    => a thermodynamically-consistent operator should drive M -> 0.
    Band-limited to max_mode (fixed, NOT nx//8) so the content stays within the
    operator's mode range and the per-step phase rotation is small -- otherwise
    high-k transport aliases and even a plain FNO fails (the operators only span
    the lowest modes_op modes)."""
    kx = torch.fft.fftfreq(nx, d=1.0/nx).to(device)
    ky = torch.fft.rfftfreq(nx, d=1.0/nx).to(device)
    KX, KY = torch.meshgrid(kx, ky, indexing='ij')
    phase = torch.exp(-1j * c * (KX + KY) * dt)
    data_in, data_out = [], []
    for _ in range(n_samples):
        n_modes = torch.randint(3, 7, (1,)).item()
        u0 = _random_field_2d(nx, nx, max_mode, n_modes, device=device)
        u_hat = torch.fft.rfft2(u0)
        traj = [u0.clone()]
        for t in range(nt):
            u_hat = u_hat * phase
            traj.append(torch.fft.irfft2(u_hat, s=(nx, nx)))
        traj = torch.stack(traj, dim=0)
        data_in.append(traj[:-1])
        data_out.append(traj[1:])
    return torch.stack(data_in), torch.stack(data_out), 'advection'


def generate_burgers_data_2d(n_samples=150, nx=128, nt=15, dt=0.002, nu=0.02, device='cpu'):
    """2D scalar Burgers: u_t + u(u_x+u_y) = nu*(uxx+uyy). Mixed rev+diss.
    Semi-implicit: diffusion in spectral, advection explicit."""
    kx = torch.fft.fftfreq(nx, d=1.0/nx).to(device)
    ky = torch.fft.rfftfreq(nx, d=1.0/nx).to(device)
    KX, KY = torch.meshgrid(kx, ky, indexing='ij')
    k_sq = KX**2 + KY**2

    data_in, data_out = [], []
    for _ in range(n_samples):
        n_modes = torch.randint(2, 5, (1,)).item()
        u = _random_field_2d(nx, nx, 5, n_modes, amp_scale=0.3, device=device)
        traj = [u.clone()]
        for t in range(nt):
            u_hat = torch.fft.rfft2(u)
            # implicit diffusion
            u_hat = u_hat / (1 + nu * k_sq * dt)
            u = torch.fft.irfft2(u_hat, s=(nx, nx))
            # explicit advection (spectral derivatives)
            ux = torch.fft.irfft2(1j * KX * torch.fft.rfft2(u), s=(nx, nx))
            uy = torch.fft.irfft2(1j * KY * torch.fft.rfft2(u), s=(nx, nx))
            u = u - dt * u * (ux + uy)
            traj.append(u.clone())
        traj = torch.stack(traj, dim=0)
        data_in.append(traj[:-1])
        data_out.append(traj[1:])
    return torch.stack(data_in), torch.stack(data_out), 'burgers'


# ============================================================================
# Building Blocks — 2D
# ============================================================================

class SpectralConv2d(nn.Module):
    """Standard FNO spectral convolution (2D). Two corners for +/- kx."""
    def __init__(self, in_ch, out_ch, modes1, modes2):
        super().__init__()
        self.modes1 = modes1  # kx modes (keep both +/- corners)
        self.modes2 = modes2  # ky modes (non-negative only, rfft)
        scale = 1.0 / (in_ch * out_ch)
        self.W1 = nn.Parameter(scale * torch.randn(out_ch, in_ch, modes1, modes2, dtype=torch.cfloat))
        self.W2 = nn.Parameter(scale * torch.randn(out_ch, in_ch, modes1, modes2, dtype=torch.cfloat))

    def forward(self, x):
        B, C, H, W = x.shape
        x_hat = torch.fft.rfft2(x, dim=(-2, -1))  # (B, C, H, W//2+1)
        out_hat = torch.zeros(B, self.W1.shape[0], H, W // 2 + 1,
                              dtype=torch.cfloat, device=x.device)
        m1 = min(self.modes1, H // 2)
        m2 = min(self.modes2, W // 2 + 1)
        # top-left corner (positive kx)
        out_hat[:, :, :m1, :m2] = torch.einsum(
            'bixy,oixy->boxy', x_hat[:, :, :m1, :m2], self.W1[:, :, :m1, :m2])
        # bottom-left corner (negative kx)
        out_hat[:, :, -m1:, :m2] = torch.einsum(
            'bixy,oixy->boxy', x_hat[:, :, -m1:, :m2], self.W2[:, :, :m1, :m2])
        return torch.fft.irfft2(out_hat, s=(H, W))


class FNO_Block2d(nn.Module):
    def __init__(self, width, modes1, modes2):
        super().__init__()
        self.conv = SpectralConv2d(width, width, modes1, modes2)
        self.skip = nn.Conv2d(width, width, 1)
        self.norm = nn.InstanceNorm2d(width)

    def forward(self, x):
        return F.gelu(self.norm(self.conv(x) + self.skip(x)))


class FNO_Backbone2d(nn.Module):
    def __init__(self, in_ch=1, out_ch=1, width=32, modes=16, n_layers=4):
        super().__init__()
        self.lift = nn.Conv2d(in_ch, width, 1)
        self.blocks = nn.ModuleList([FNO_Block2d(width, modes, modes) for _ in range(n_layers)])
        self.proj = nn.Sequential(
            nn.Conv2d(width, width, 1),
            nn.GELU(),
            nn.Conv2d(width, out_ch, 1)
        )

    def forward(self, x):
        x = self.lift(x)
        for block in self.blocks:
            x = block(x)
        return self.proj(x)


class FunctionalNet2d(nn.Module):
    """FNO2d backbone → scalar functional F[u]. u:(B,1,H,W) → (B,)."""
    def __init__(self, width=24, modes=12, n_layers=3):
        super().__init__()
        self.backbone = FNO_Backbone2d(in_ch=1, out_ch=1, width=width,
                                        modes=modes, n_layers=n_layers)
        self.head = nn.Sequential(
            nn.Linear(1, 16),
            nn.GELU(),
            nn.Linear(16, 1)
        )

    def forward(self, u):
        density = self.backbone(u)              # (B,1,H,W)
        integral = density.mean(dim=(-1, -2))   # (B,1) — spatial average ∝ integral
        return self.head(integral).squeeze(-1)  # (B,)


# ============================================================================
# Model 1: Vanilla FNO (2D, residual)
# ============================================================================

class VanillaFNO2d(nn.Module):
    def __init__(self, width=32, modes=16, n_layers=4):
        super().__init__()
        self.backbone = FNO_Backbone2d(in_ch=1, out_ch=1, width=width,
                                        modes=modes, n_layers=n_layers)

    def forward(self, u):
        return u + self.backbone(u)

    def predict_with_info(self, u):
        return self.forward(u), {}


# ============================================================================
# Model 2: EP-FNO (2D, energy penalty)
# ============================================================================

class EP_FNO2d(nn.Module):
    def __init__(self, width=32, modes=16, n_layers=4):
        super().__init__()
        self.backbone = FNO_Backbone2d(in_ch=1, out_ch=1, width=width,
                                        modes=modes, n_layers=n_layers)

    def forward(self, u):
        return u + self.backbone(u)

    def predict_with_info(self, u):
        u_next = self.forward(u)
        E_in = 0.5 * (u**2).mean(dim=(-1, -2)).mean(dim=-1)
        E_out = 0.5 * (u_next**2).mean(dim=(-1, -2)).mean(dim=-1)
        return u_next, {'dE': E_out - E_in}

    def energy_penalty(self, info, pde_type):
        dE = info['dE']
        if pde_type in ('heat', 'burgers'):
            return (F.relu(dE)**2).mean()
        elif pde_type in ('wave', 'advection'):
            return (dE**2).mean()
        return torch.tensor(0.0, device=dE.device)


# ============================================================================
# Model 3: GENERIC-FNO (2D)
# ============================================================================

class GENERIC_FNO2d(nn.Module):
    """
    du/dt = L·δE/δu + M·δS/δu  with hard projection.
    L(kx,ky) = i·a  (anti-Hermitian diagonal), M(kx,ky) = |b|² (PSD diagonal).
    Operators are defined on the lowest (modes_op) modes in each direction,
    using two corners for +/- kx (rfft2 layout) → resolution invariant.
    """
    def __init__(self, nx=128, width_func=24, modes_func=12, n_layers_func=3,
                 modes_op=16, residual_gate_init=-3.0, l2_vargrad=False,
                 use_residual=True, degeneracy_construction=True):
        super().__init__()
        self.nx = nx
        self.modes_op = modes_op
        # degeneracy_construction (DEFAULT, the thermodynamically-consistent model):
        #   build L = (I-P_S) D_L (I-P_S) and M = (I-P_E) D_M (I-P_E), where P_E,P_S
        #   are rank-1 projections onto delta E/delta u, delta S/delta u and D_L=i*a,
        #   D_M=|b|^2 are diagonal Fourier multipliers. Then L dS = 0 and M dE = 0
        #   EXACTLY, so energy is conserved (dE/dt=0) and entropy is produced
        #   (dS/dt=<dS,M dS> >= 0) by construction in ANY dimension -- no energy
        #   projection, no entropy correction, no free residual. A reversible PDE
        #   (wave) is forced to learn M->0 because dissipation can no longer hide
        #   behind a projection. Set False for the legacy projection+correction path
        #   (kept only for ablation; it does NOT specialize -- M and L are
        #   interchangeable under the projection, so everything routes through M).
        # l2_vargrad: use the L2 variational derivative (N/|Omega|) grad_u E. Affects
        #   only the operator-output SCALE (projections are scale-invariant ratios);
        #   harmless either way under the construction. Default off.
        # use_residual: only meaningful in the legacy path; a free residual would
        #   break the thermodynamic guarantee, so it is ignored when
        #   degeneracy_construction=True.
        self.degeneracy_construction = degeneracy_construction
        self.l2_vargrad = l2_vargrad
        self.use_residual = use_residual
        m1 = min(modes_op, nx // 2)
        m2 = min(modes_op, nx // 2 + 1)
        self.m1, self.m2 = m1, m2

        self.E_net = FunctionalNet2d(width=width_func, modes=modes_func, n_layers=n_layers_func)
        self.S_net = FunctionalNet2d(width=width_func, modes=modes_func, n_layers=n_layers_func)

        # L: anti-Hermitian diagonal, two kx corners
        self.a_pos = nn.Parameter(0.3 * torch.randn(m1, m2))
        self.a_neg = nn.Parameter(0.3 * torch.randn(m1, m2))
        # M: PSD diagonal, two kx corners (parameterized as |b|²)
        self.b_pos_r = nn.Parameter(0.3 * torch.randn(m1, m2))
        self.b_pos_i = nn.Parameter(0.3 * torch.randn(m1, m2))
        self.b_neg_r = nn.Parameter(0.3 * torch.randn(m1, m2))
        self.b_neg_i = nn.Parameter(0.3 * torch.randn(m1, m2))

        # Small gated residual for high-freq content
        self.residual = nn.Sequential(
            nn.Conv2d(1, 16, 1),
            nn.GELU(),
            nn.Conv2d(16, 1, 1)
        )
        self.residual_gate = nn.Parameter(torch.tensor(float(residual_gate_init)))

    def _apply_operators(self, dEdu_hat, dSdu_hat, H, W):
        """Apply diagonal L and M in 2D Fourier space (two kx corners)."""
        m1, m2 = self.m1, self.m2
        rev_hat = torch.zeros_like(dEdu_hat)
        diss_hat = torch.zeros_like(dSdu_hat)

        # L = i*a  (reversible)
        rev_hat[:, :, :m1, :m2] = 1j * self.a_pos * dEdu_hat[:, :, :m1, :m2]
        rev_hat[:, :, -m1:, :m2] = 1j * self.a_neg * dEdu_hat[:, :, -m1:, :m2]

        # M = |b|²  (dissipative, PSD)
        M_pos = self.b_pos_r**2 + self.b_pos_i**2
        M_neg = self.b_neg_r**2 + self.b_neg_i**2
        diss_hat[:, :, :m1, :m2] = M_pos * dSdu_hat[:, :, :m1, :m2]
        diss_hat[:, :, -m1:, :m2] = M_neg * dSdu_hat[:, :, -m1:, :m2]

        return rev_hat, diss_hat

    # --- single-operator Fourier multipliers (for degeneracy-by-construction) ---
    def _L_apply(self, v, H, W):
        """Apply the skew diagonal operator D_L = i*a to physical field v."""
        m1, m2 = self.m1, self.m2
        vh = torch.fft.rfft2(v, dim=(-2, -1))
        out = torch.zeros_like(vh)
        out[:, :, :m1, :m2] = 1j * self.a_pos * vh[:, :, :m1, :m2]
        out[:, :, -m1:, :m2] = 1j * self.a_neg * vh[:, :, -m1:, :m2]
        return torch.fft.irfft2(out, s=(H, W))

    def _M_apply(self, v, H, W):
        """Apply the PSD diagonal operator D_M = |b|^2 to physical field v."""
        m1, m2 = self.m1, self.m2
        vh = torch.fft.rfft2(v, dim=(-2, -1))
        out = torch.zeros_like(vh)
        Mp = self.b_pos_r**2 + self.b_pos_i**2
        Mn = self.b_neg_r**2 + self.b_neg_i**2
        out[:, :, :m1, :m2] = Mp * vh[:, :, :m1, :m2]
        out[:, :, -m1:, :m2] = Mn * vh[:, :, -m1:, :m2]
        return torch.fft.irfft2(out, s=(H, W))

    @staticmethod
    def _remove(v, w):
        """(I - P_w) v: remove the component of v along direction w, per sample.
        Scale-invariant in w (ratio), so the L2-vs-Euclidean choice is irrelevant."""
        ip = (v * w).sum(dim=(-1, -2), keepdim=True)
        nn = (w * w).sum(dim=(-1, -2), keepdim=True) + 1e-12
        return v - (ip / nn) * w

    def _generic_rhs(self, dEdu, dSdu, H, W):
        """du/dt = (I-P_S) D_L (I-P_S) dE + (I-P_E) D_M (I-P_E) dS.
        Degeneracy (L dS = 0, M dE = 0) holds exactly => dE/dt = 0 and
        dS/dt = <dS, M dS> >= 0 by construction, no projection needed."""
        rev = self._remove(self._L_apply(self._remove(dEdu, dSdu), H, W), dSdu)
        diss = self._remove(self._M_apply(self._remove(dSdu, dEdu), H, W), dEdu)
        return rev, diss

    def forward(self, u):
        B, C, H, W = u.shape
        u_leaf = u.detach().requires_grad_(True)

        E = self.E_net(u_leaf)
        S = self.S_net(u_leaf)
        dEdu = torch.autograd.grad(E.sum(), u_leaf, create_graph=True)[0]
        dSdu = torch.autograd.grad(S.sum(), u_leaf, create_graph=True)[0]

        if self.l2_vargrad:
            # L2 variational derivative: delta E/delta u = (N/|Omega|) grad_u E.
            scale = (H * W) / (2.0 * math.pi) ** 2
            dEdu = dEdu * scale
            dSdu = dSdu * scale

        if self.degeneracy_construction:
            # Thermodynamically-consistent path: degeneracy by construction.
            rev, diss = self._generic_rhs(dEdu, dSdu, H, W)
            dudt = rev + diss
            return u + dudt

        # --- legacy projection + correction path (ablation only) ---
        dEdu_hat = torch.fft.rfft2(dEdu, dim=(-2, -1))
        dSdu_hat = torch.fft.rfft2(dSdu, dim=(-2, -1))

        rev_hat, diss_hat = self._apply_operators(dEdu_hat, dSdu_hat, H, W)
        rev = torch.fft.irfft2(rev_hat, s=(H, W))
        diss = torch.fft.irfft2(diss_hat, s=(H, W))

        dudt = rev + diss
        dudt = self._project_energy_conservation(dudt, dEdu)
        dudt = self._ensure_entropy_production(dudt, dSdu, dEdu)

        if self.use_residual:
            gate = torch.sigmoid(self.residual_gate)
            residual = gate * self.residual(u_leaf)
            residual = self._project_energy_conservation(residual, dEdu)
            dudt = dudt + residual

        return u + dudt

    def _project_energy_conservation(self, dudt, dEdu):
        """Project du/dt ⊥ δE/δu over both spatial dims → dE/dt = 0."""
        inner = (dudt * dEdu).sum(dim=(-1, -2), keepdim=True)
        norm_sq = (dEdu * dEdu).sum(dim=(-1, -2), keepdim=True) + 1e-10
        return dudt - (inner / norm_sq) * dEdu

    def _ensure_entropy_production(self, dudt, dSdu, dEdu):
        """Ensure dS/dt = ⟨δS/δu, du/dt⟩ ≥ 0 via correction ⊥ δE/δu."""
        dSdt = (dSdu * dudt).sum(dim=(-1, -2), keepdim=True)
        violation = F.relu(-dSdt)
        if violation.sum() > 0:
            inner_SE = (dSdu * dEdu).sum(dim=(-1, -2), keepdim=True)
            norm_E_sq = (dEdu * dEdu).sum(dim=(-1, -2), keepdim=True) + 1e-10
            dSdu_perp = dSdu - (inner_SE / norm_E_sq) * dEdu
            inner_S_Sperp = (dSdu * dSdu_perp).sum(dim=(-1, -2), keepdim=True) + 1e-10
            alpha = violation / inner_S_Sperp
            dudt = dudt + alpha * dSdu_perp
        return dudt

    def entropy_production(self, u):
        """Normalized entropy production r_S = <dS/du, du/dt> / (||dS/du|| ||du/dt||),
        fully differentiable. Used as a minimum-entropy-production (MEP) penalty:
        among GENERIC representations consistent with the data, prefer the one that
        produces the least entropy. Scale-invariant in S (can't be gamed by
        rescaling S), so reducing it requires genuinely making du/dt more
        orthogonal to dS/du -- i.e. genuinely less dissipative."""
        u_leaf = u.detach().requires_grad_(True)
        E = self.E_net(u_leaf); S = self.S_net(u_leaf)
        dEdu = torch.autograd.grad(E.sum(), u_leaf, create_graph=True)[0]
        dSdu = torch.autograd.grad(S.sum(), u_leaf, create_graph=True)[0]
        H, W = u.shape[-2], u.shape[-1]
        if self.l2_vargrad:
            scale = (H * W) / (2.0 * math.pi) ** 2
            dEdu = dEdu * scale
            dSdu = dSdu * scale
        if self.degeneracy_construction:
            rev, diss = self._generic_rhs(dEdu, dSdu, H, W)
            dudt = rev + diss
        else:
            dEh = torch.fft.rfft2(dEdu, dim=(-2, -1))
            dSh = torch.fft.rfft2(dSdu, dim=(-2, -1))
            rev_hat, diss_hat = self._apply_operators(dEh, dSh, H, W)
            rev = torch.fft.irfft2(rev_hat, s=(H, W))
            diss = torch.fft.irfft2(diss_hat, s=(H, W))
            dudt = rev + diss
            dudt = self._project_energy_conservation(dudt, dEdu)
            dudt = self._ensure_entropy_production(dudt, dSdu, dEdu)
            if self.use_residual:
                gate = torch.sigmoid(self.residual_gate)
                res = self._project_energy_conservation(gate * self.residual(u_leaf), dEdu)
                dudt = dudt + res
        num = (dSdu * dudt).flatten(1).sum(dim=1)
        den = dSdu.flatten(1).norm(dim=1) * dudt.flatten(1).norm(dim=1) + 1e-8
        return (num / den).clamp(min=0).mean()

    def predict_with_info(self, u):
        u_leaf = u.detach().requires_grad_(True)
        E = self.E_net(u_leaf)
        S = self.S_net(u_leaf)
        _ = torch.autograd.grad(E.sum(), u_leaf, create_graph=True)[0]
        _ = torch.autograd.grad(S.sum(), u_leaf, create_graph=True)[0]
        u_next = self.forward(u)
        u_next_leaf = u_next.detach().requires_grad_(True)
        E_next = self.E_net(u_next_leaf)
        S_next = self.S_net(u_next_leaf)
        info = {
            'E': E.detach(), 'S': S.detach(),
            'dE': (E_next - E).detach(), 'dS': (S_next - S).detach(),
        }
        return u_next, info

    def degeneracy_loss(self, u):
        """Soft regularizer: L·δS/δu ≈ 0 and M·δE/δu ≈ 0.
        Under degeneracy_construction these hold exactly, so the penalty is 0
        (kept only for the legacy projection path)."""
        if self.degeneracy_construction:
            return torch.zeros((), device=u.device)
        u_leaf = u.detach().requires_grad_(True)
        E = self.E_net(u_leaf)
        S = self.S_net(u_leaf)
        dEdu = torch.autograd.grad(E.sum(), u_leaf, create_graph=True)[0]
        dSdu = torch.autograd.grad(S.sum(), u_leaf, create_graph=True)[0]
        dEdu_hat = torch.fft.rfft2(dEdu, dim=(-2, -1))
        dSdu_hat = torch.fft.rfft2(dSdu, dim=(-2, -1))
        m1, m2 = self.m1, self.m2

        # L·δS/δu
        L_dS_pos = 1j * self.a_pos * dSdu_hat[:, :, :m1, :m2]
        L_dS_neg = 1j * self.a_neg * dSdu_hat[:, :, -m1:, :m2]
        loss_L = (L_dS_pos.abs()**2).mean() + (L_dS_neg.abs()**2).mean()

        # M·δE/δu
        M_pos = self.b_pos_r**2 + self.b_pos_i**2
        M_neg = self.b_neg_r**2 + self.b_neg_i**2
        M_dE_pos = M_pos * dEdu_hat[:, :, :m1, :m2]
        M_dE_neg = M_neg * dEdu_hat[:, :, -m1:, :m2]
        loss_M = (M_dE_pos.abs()**2).mean() + (M_dE_neg.abs()**2).mean()

        return loss_L + loss_M


# ============================================================================
# Training
# ============================================================================


# ============================================================================
# Evaluation
# ============================================================================

def evaluate_model(model, data_in, data_out, pde_type, model_type='fno',
                   device='cpu', n_rollout=10, eval_batch=4):
    """Memory-safe evaluation: keeps test data on CPU, moves only small chunks
    to GPU. The GENERIC-FNO autograd graph (for delta E/delta u) is large at high
    resolution, so we process eval_batch samples at a time and free between chunks."""
    model.eval()
    model = model.to(device)

    n_test = min(40, data_in.shape[0])
    nt = min(n_rollout, data_in.shape[1])
    is_generic = (model_type == 'generic')

    l2_single = []
    rollout_err_per_t = [[] for _ in range(nt)]
    E_pred_per_t = [[] for _ in range(nt)]
    E_true_per_t = [[] for _ in range(nt)]
    E_init_list, dE_list, dS_list = [], [], []

    for start in range(0, n_test, eval_batch):
        end = min(start + eval_batch, n_test)
        x0 = data_in[start:end, 0:1, :, :].to(device)
        y0 = data_out[start:end, 0:1, :, :].to(device)

        # single-step
        with torch.enable_grad():
            pred = model(x0).detach()
        l2 = ((pred - y0)**2).mean(dim=(-1, -2)).sqrt() / ((y0**2).mean(dim=(-1, -2)).sqrt() + 1e-8)
        l2_single.append(l2.flatten().cpu())

        # structural metrics (GENERIC) — same starting state, cheap
        if is_generic:
            with torch.enable_grad():
                _, info = model.predict_with_info(x0)
            dE_list.append(info['dE'].flatten().cpu())
            dS_list.append(info['dS'].flatten().cpu())

        # rollout
        E_init_list.append((0.5 * (x0**2).mean(dim=(-1, -2, -3))).cpu())
        x_roll = x0.clone()
        for t in range(nt):
            with torch.enable_grad():
                x_roll = model(x_roll).detach()
            y_t = data_out[start:end, t:t+1, :, :].to(device)
            err = ((x_roll - y_t)**2).mean(dim=(-1, -2)).sqrt() / ((y_t**2).mean(dim=(-1, -2)).sqrt() + 1e-8)
            rollout_err_per_t[t].append(err.flatten().cpu())
            E_pred_per_t[t].append((0.5 * (x_roll**2).mean(dim=(-1, -2, -3))).cpu())
            E_true_per_t[t].append((0.5 * (y_t**2).mean(dim=(-1, -2, -3))).cpu())
            del y_t

        del x0, y0, x_roll, pred
        if device == 'cuda':
            torch.cuda.empty_cache()

    results = {}
    results['l2_single'] = torch.cat(l2_single).mean().item()
    rollout_means = [torch.cat(r).mean().item() for r in rollout_err_per_t]
    results['l2_rollout'] = float(np.mean(rollout_means))

    E_pred_stack = torch.stack([torch.cat(e) for e in E_pred_per_t], dim=1)  # (n_test, nt)
    E_true_stack = torch.stack([torch.cat(e) for e in E_true_per_t], dim=1)
    results['energy_track'] = ((E_pred_stack - E_true_stack)**2).mean().sqrt().item()

    E_init = torch.cat(E_init_list)  # (n_test,)
    if pde_type in ('heat', 'burgers'):
        E_all = torch.cat([E_init.unsqueeze(1), E_pred_stack], dim=1)
        dE = E_all[:, 1:] - E_all[:, :-1]
        results['mono_violations'] = (dE > 1e-6).float().mean().item() * 100
    elif pde_type in ('wave', 'advection'):
        E_all = torch.cat([E_init.unsqueeze(1), E_pred_stack], dim=1)
        results['mono_violations'] = E_all.std(dim=1).mean().item()
    else:
        results['mono_violations'] = 0.0

    if is_generic:
        results['dE_mean'] = torch.cat(dE_list).mean().item()
        results['dS_mean'] = torch.cat(dS_list).mean().item()
    else:
        results['dE_mean'] = 0.0
        results['dS_mean'] = 0.0

    return results


# ============================================================================
# Main Benchmark
# ============================================================================

def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

# ============================================================================
# Band-limited data generators (fixed max_mode → resolution-independent physics)
# ============================================================================

def gen_heat_bl(n_samples, nx, nt=15, dt=0.005, nu=0.02, max_mode=8, device='cpu'):
    kx = torch.fft.fftfreq(nx, d=1.0/nx).to(device)
    ky = torch.fft.rfftfreq(nx, d=1.0/nx).to(device)
    KX, KY = torch.meshgrid(kx, ky, indexing='ij')
    decay = torch.exp(-nu * (KX**2 + KY**2) * dt)
    di, do = [], []
    for _ in range(n_samples):
        nm = torch.randint(3, 7, (1,)).item()
        u0 = _random_field_2d(nx, nx, max_mode, nm, device=device)
        uh = torch.fft.rfft2(u0)
        traj = [u0.clone()]
        for _ in range(nt):
            uh = uh * decay
            traj.append(torch.fft.irfft2(uh, s=(nx, nx)))
        traj = torch.stack(traj, 0)
        di.append(traj[:-1]); do.append(traj[1:])
    return torch.stack(di), torch.stack(do), 'heat'


def gen_wave_bl(n_samples, nx, nt=15, dt=0.005, c=1.0, max_mode=8, device='cpu'):
    kx = torch.fft.fftfreq(nx, d=1.0/nx).to(device)
    ky = torch.fft.rfftfreq(nx, d=1.0/nx).to(device)
    KX, KY = torch.meshgrid(kx, ky, indexing='ij')
    omega = c * torch.sqrt(KX**2 + KY**2)
    safe = torch.where(omega > 1e-8, omega, torch.ones_like(omega))
    di, do = [], []
    for _ in range(n_samples):
        nm = torch.randint(3, 7, (1,)).item()
        u0 = _random_field_2d(nx, nx, max_mode, nm, device=device)
        v0 = _random_field_2d(nx, nx, max_mode, nm, amp_scale=0.3, device=device)
        uh = torch.fft.rfft2(u0); vh = torch.fft.rfft2(v0)
        traj = [u0.clone()]
        for _ in range(nt):
            cw, sw = torch.cos(omega*dt), torch.sin(omega*dt)
            un = cw*uh + (sw/safe)*vh
            vn = -safe*sw*uh + cw*vh
            uh, vh = un, vn
            traj.append(torch.fft.irfft2(uh, s=(nx, nx)))
        traj = torch.stack(traj, 0)
        di.append(traj[:-1]); do.append(traj[1:])
    return torch.stack(di), torch.stack(do), 'wave'


def gen_advection_bl(n_samples, nx, nt=15, dt=0.02, c=1.0, max_mode=8, device='cpu'):
    """2D linear advection u_t + c(u_x+u_y)=0. Reversible, Markovian in u,
    conserves 0.5<u^2> exactly. The clean scalar reversible test => M should -> 0."""
    kx = torch.fft.fftfreq(nx, d=1.0/nx).to(device)
    ky = torch.fft.rfftfreq(nx, d=1.0/nx).to(device)
    KX, KY = torch.meshgrid(kx, ky, indexing='ij')
    phase = torch.exp(-1j * c * (KX + KY) * dt)
    di, do = [], []
    for _ in range(n_samples):
        nm = torch.randint(3, 7, (1,)).item()
        u0 = _random_field_2d(nx, nx, max_mode, nm, device=device)
        uh = torch.fft.rfft2(u0)
        traj = [u0.clone()]
        for _ in range(nt):
            uh = uh * phase
            traj.append(torch.fft.irfft2(uh, s=(nx, nx)))
        traj = torch.stack(traj, 0)
        di.append(traj[:-1]); do.append(traj[1:])
    return torch.stack(di), torch.stack(do), 'advection'


def gen_burgers_bl(n_samples, nx, nt=15, dt=0.002, nu=0.02, max_mode=5, device='cpu'):
    kx = torch.fft.fftfreq(nx, d=1.0/nx).to(device)
    ky = torch.fft.rfftfreq(nx, d=1.0/nx).to(device)
    KX, KY = torch.meshgrid(kx, ky, indexing='ij')
    k_sq = KX**2 + KY**2
    di, do = [], []
    for _ in range(n_samples):
        nm = torch.randint(2, 5, (1,)).item()
        u = _random_field_2d(nx, nx, max_mode, nm, amp_scale=0.3, device=device)
        traj = [u.clone()]
        for _ in range(nt):
            uh = torch.fft.rfft2(u) / (1 + nu * k_sq * dt)
            u = torch.fft.irfft2(uh, s=(nx, nx))
            ux = torch.fft.irfft2(1j*KX*torch.fft.rfft2(u), s=(nx, nx))
            uy = torch.fft.irfft2(1j*KY*torch.fft.rfft2(u), s=(nx, nx))
            u = u - dt * u * (ux + uy)
            traj.append(u.clone())
        traj = torch.stack(traj, 0)
        di.append(traj[:-1]); do.append(traj[1:])
    return torch.stack(di), torch.stack(do), 'burgers'


GEN_BL = {'heat': gen_heat_bl, 'wave': gen_wave_bl, 'burgers': gen_burgers_bl,
          'advection': gen_advection_bl}


# ============================================================================
# Training (default: NO E-supervision; structure alone drives the result)
# ============================================================================

def train_model(model, data_in, data_out, pde_type, model_type='fno',
                n_epochs=120, lr=1e-3, batch_size=16, device='cpu',
                e_sup_mode='none', min_diss_weight=0.0,
                degeneracy_weight=0.01, residual_penalty=0.0):
    """e_sup_mode (GENERIC only):
        'none'    : no functional supervision  <-- DEFAULT. Conservation of the
                    learned E is purely architectural (the projection).
        'half_u2' : warm-start E toward 0.5<u^2>  (legacy; drags E toward a
                    decaying target — kept only for ablation).
        'mean_u'  : warm-start E toward <u>.
        's_gauge' : light, persistent anchor of S toward -0.5<u^2> so the
                    entropy consistently tracks dissipation. E is left FREE,
                    so energy conservation remains structural. Use this to get
                    a clean, interpretable S (for figures).
    degeneracy_weight : weight on ||L dS||^2 + ||M dE||^2 (default 0.01).
    residual_penalty  : weight on sigmoid(residual_gate); >0 keeps the gated
                    residual small so the GENERIC operators must carry the
                    dynamics and therefore specialize (L->0 for reversible
                    modes, M->0 for conservative PDEs). Default 0 (off)."""
    model = model.to(device)
    data_in = data_in.to(device)
    data_out = data_out.to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, n_epochs)

    n_samples = data_in.shape[0]
    nt = data_in.shape[1]

    for epoch in range(n_epochs):
        model.train()
        perm = torch.randperm(n_samples, device=device)
        epoch_losses = defaultdict(float)
        n_batches = 0

        for i in range(0, n_samples, batch_size):
            idx = perm[i:i+batch_size]
            t_idx = torch.randint(0, nt, (1,)).item()
            x = data_in[idx, t_idx:t_idx+1, :, :].clone()
            y = data_out[idx, t_idx:t_idx+1, :, :].clone()

            if epoch > 25 and torch.rand(1).item() < 0.25:
                t_start = torch.randint(0, max(1, nt-2), (1,)).item()
                x = data_in[idx, t_start:t_start+1, :, :].clone()
                rollout_len = min(2, nt - t_start)
                pred = x
                rollout_loss = 0
                for step in range(rollout_len):
                    pred = model(pred)
                    target = data_out[idx, t_start+step:t_start+step+1, :, :]
                    rollout_loss += F.mse_loss(pred, target)
                loss = rollout_loss / rollout_len
                epoch_losses['rollout'] += loss.item()
                if model_type == 'generic':
                    deg = model.degeneracy_loss(x)
                    loss += min(1.0, epoch / 50.0) * degeneracy_weight * deg
                    epoch_losses['degeneracy'] += deg.item()
                    if residual_penalty > 0.0:
                        loss = loss + residual_penalty * torch.sigmoid(model.residual_gate)
                optimizer.zero_grad(); loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step(); n_batches += 1
                continue

            pred = model(x)
            loss = F.mse_loss(pred, y)
            epoch_losses['data'] += loss.item()

            if model_type == 'ep-fno':
                _, info = model.predict_with_info(x)
                ep = model.energy_penalty(info, pde_type)
                loss += min(1.0, epoch / 30.0) * 0.1 * ep
                epoch_losses['energy_penalty'] += ep.item()

            elif model_type == 'generic':
                deg = model.degeneracy_loss(x)
                loss += min(1.0, epoch / 50.0) * degeneracy_weight * deg
                epoch_losses['degeneracy'] += deg.item()
                if residual_penalty > 0.0:
                    loss = loss + residual_penalty * torch.sigmoid(model.residual_gate)
                    epoch_losses['res_gate'] += float(torch.sigmoid(model.residual_gate))

                if min_diss_weight > 0.0:
                    mep = model.entropy_production(x)
                    loss += min(1.0, epoch / 50.0) * min_diss_weight * mep
                    epoch_losses['min_diss'] += mep.item()

                if e_sup_mode in ('half_u2', 'mean_u'):
                    u_leaf = x.detach().requires_grad_(True)
                    E_pred = model.E_net(u_leaf)
                    if e_sup_mode == 'half_u2':
                        E_true = 0.5 * (x**2).mean(dim=(-1, -2, -3))
                    else:
                        E_true = x.mean(dim=(-1, -2, -3))
                    e_sup = F.mse_loss(E_pred, E_true)
                    loss += max(0, 1.0 - epoch / 80.0) * 0.1 * e_sup
                    epoch_losses['E_supervision'] += e_sup.item()

                elif e_sup_mode == 's_gauge':
                    # Gauge fix: align the dissipative DIRECTION delta S/delta u with -u
                    # (cosine), i.e. S ~ -0.5 int u^2 so entropy rises as mechanical
                    # energy falls. Scale-invariant (won't fight M); E stays free.
                    u_leaf = x.detach().requires_grad_(True)
                    S_pred = model.S_net(u_leaf)
                    dSdu = torch.autograd.grad(S_pred.sum(), u_leaf,
                                               create_graph=True)[0]
                    tgt = -u_leaf
                    num = (dSdu * tgt).sum(dim=(-1, -2, -3))
                    den = (dSdu.flatten(1).norm(dim=1) *
                           tgt.flatten(1).norm(dim=1) + 1e-8)
                    gauge_loss = (1.0 - num / den).mean()
                    loss += 0.2 * gauge_loss
                    epoch_losses['S_gauge'] += gauge_loss.item()
                # 'none': no functional supervision

            optimizer.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step(); n_batches += 1

        scheduler.step()
        if (epoch + 1) % 30 == 0 or epoch == 0:
            avg = {k: v / max(n_batches, 1) for k, v in epoch_losses.items()}
            extras = ' '.join(f"{k}={v:.6f}" for k, v in avg.items())
            print(f"  Epoch {epoch+1:3d}: {extras}")

    return model


# ============================================================================
# Probe: what did E-net and S-net learn (along ground-truth trajectories)?
# ============================================================================

def _pearson(a, b):
    a = a - a.mean(); b = b - b.mean()
    return float((a * b).sum() / (a.norm() * b.norm() + 1e-12))


def probe_functionals(model, data_in, pde_type, device='cpu', n_probe=24):
    model.eval().to(device)
    N = min(n_probe, data_in.shape[0])
    E0, meanu0, halfu20 = [], [], []
    drift_ratios, corr_S, mechE_decays = [], [], []
    with torch.no_grad():
        for i in range(N):
            u = data_in[i].to(device).unsqueeze(1)   # (T,1,H,W)
            Ev = model.E_net(u).cpu(); Sv = model.S_net(u).cpu()
            mean_u = u.mean(dim=(-1, -2, -3)).cpu()
            half_u2 = 0.5 * (u**2).mean(dim=(-1, -2, -3)).cpu()
            E0.append(Ev[0]); meanu0.append(mean_u[0]); halfu20.append(half_u2[0])
            drift_ratios.append(float(Ev.std() / (half_u2.std() + 1e-10)))
            corr_S.append(_pearson(Sv, half_u2))
            mechE_decays.append(float((half_u2[0]-half_u2[-1])/(half_u2[0].abs()+1e-10)))
            del u
            if device == 'cuda':
                torch.cuda.empty_cache()
    E0 = torch.stack(E0); meanu0 = torch.stack(meanu0); halfu20 = torch.stack(halfu20)
    return {'E_drift_ratio': float(np.mean(drift_ratios)),
            'corr_E_meanu': _pearson(E0, meanu0),
            'corr_E_halfu2': _pearson(E0, halfu20),
            'corr_S_halfu2': float(np.mean(corr_S)),
            'mechE_decay': float(np.mean(mechE_decays))}


# ============================================================================
# GENERATOR-CHANNEL DIAGNOSTIC (scale-invariant L vs M usage)
# ============================================================================

def channel_diagnostics(model, X, n_batch=8, Y=None):
    """Scale-invariant test of how much a trained GENERIC-FNO2d routes through the
    dissipative (M) channel, plus a GAUGE-INVARIANT dissipation measure.

      rho_M = ||diss|| / (||rev|| + ||diss||)   M-channel fraction (gauge-dependent).
      r_S   = <dS/du, du/dt> / (||dS/du|| ||du/dt||)   learned-entropy production
              (scale-invariant in S but NOT gauge-invariant: shifts with the
              (E,S) representation; reported for completeness).
      r_E   = |<dE/du, du/dt>| / (||dE/du|| ||du/dt||)  energy-projection check.

      GAUGE-INVARIANT (independent of the learned E,S -- uses only the predicted
      dynamics and the FIXED physical quadratic energy Q[u]=0.5||u||^2):
      r_mech = -<u, du/dt> / (||u|| ||du/dt||)   normalized dissipation rate of Q.
               >0 dissipating, ~0 conserving (reversible), <0 spurious injection.
      pi_model = (Q(u) - Q(model(u))) / Q(u)     relative Q dissipated per step.
      pi_true  = (Q(u) - Q(Y)) / Q(u)            same for ground truth (if Y given);
               pi_model ~ pi_true validates the physical dissipation rate.
      L_mag, M_mag : RMS magnitude of the learned multipliers.
    """
    device = next(model.parameters()).device
    model.eval()
    X = X[:n_batch].to(device).detach()
    H, W = X.shape[-2], X.shape[-1]

    u_leaf = X.clone().requires_grad_(True)
    E = model.E_net(u_leaf); S = model.S_net(u_leaf)
    dEdu = torch.autograd.grad(E.sum(), u_leaf, retain_graph=True)[0].detach()
    dSdu = torch.autograd.grad(S.sum(), u_leaf)[0].detach()

    dEh = torch.fft.rfft2(dEdu, dim=(-2, -1))
    dSh = torch.fft.rfft2(dSdu, dim=(-2, -1))
    if getattr(model, 'degeneracy_construction', False):
        rev, diss = model._generic_rhs(dEdu, dSdu, H, W)
        rev = rev.detach(); diss = diss.detach()
    else:
        rev_hat, diss_hat = model._apply_operators(dEh, dSh, H, W)
        rev = torch.fft.irfft2(rev_hat.detach(), s=(H, W))
        diss = torch.fft.irfft2(diss_hat.detach(), s=(H, W))

    a2 = model.a_pos.detach() ** 2 + model.a_neg.detach() ** 2
    Mp = model.b_pos_r.detach() ** 2 + model.b_pos_i.detach() ** 2
    Mn = model.b_neg_r.detach() ** 2 + model.b_neg_i.detach() ** 2
    L_mag = a2.mean().sqrt().item()
    M_mag = ((Mp ** 2 + Mn ** 2).mean()).sqrt().item()

    def pnorm(z): return z.flatten(1).norm(dim=1)
    def ip(a, b): return (a * b).flatten(1).sum(dim=1)
    nrev, ndiss = pnorm(rev), pnorm(diss)
    rho_M = (ndiss / (nrev + ndiss + 1e-12)).mean().item()

    Xn = model(X).detach()
    dudt = (Xn - X).detach()
    r_S = (ip(dSdu, dudt) / (pnorm(dSdu) * pnorm(dudt) + 1e-12)).mean().item()
    r_E = (ip(dEdu, dudt).abs() / (pnorm(dEdu) * pnorm(dudt) + 1e-12)).mean().item()

    # gauge-invariant dissipation of the fixed quadratic energy Q = 0.5||u||^2
    r_mech = (-ip(X, dudt) / (pnorm(X) * pnorm(dudt) + 1e-12)).mean().item()
    Qx = (X ** 2).flatten(1).sum(dim=1)
    pi_model = ((Qx - (Xn ** 2).flatten(1).sum(dim=1)) / (Qx + 1e-12)).mean().item()
    out = {'rho_M': rho_M, 'r_S': r_S, 'r_E': r_E, 'L_mag': L_mag, 'M_mag': M_mag,
           'r_mech': r_mech, 'pi_model': pi_model}
    if Y is not None:
        Yb = Y[:n_batch].to(device).detach()
        out['pi_true'] = ((Qx - (Yb ** 2).flatten(1).sum(dim=1)) / (Qx + 1e-12)).mean().item()
    return out


# ============================================================================
# (a) HEADLINE BENCHMARK — default e_sup_mode='none'
#     (saves GENERIC checkpoints + runs the channel diagnostic in one pass)
# ============================================================================

def run_benchmark(save_dir=None, nx=128, n_samples=150, nt=15, n_epochs=120,
                  min_diss_weight=0.0, l2_vargrad=False, use_residual=True,
                  degeneracy_construction=True, e_sup_mode='none',
                  pdes=('heat', 'advection', 'burgers', 'wave')):
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print("=" * 80)
    print(f"GENERIC-FNO 2D BENCHMARK (no E-supervision)  |  device={device}")
    print("=" * 80)
    BATCH = 16
    WIDTH, MODES, N_LAYERS = 32, 16, 4
    WIDTH_FUNC, MODES_FUNC, N_LAYERS_FUNC = 24, 12, 3
    GENS = {'heat': generate_heat_data_2d, 'wave': generate_wave_data_2d,
            'advection': generate_advection_data_2d,
            'burgers': generate_burgers_data_2d}
    GENS = {k: GENS[k] for k in pdes}

    print(f"Config: nx={nx} n_samples={n_samples} nt={nt} epochs={n_epochs} "
          f"min_diss_weight={min_diss_weight} l2_vargrad={l2_vargrad} "
          f"use_residual={use_residual} degeneracy_construction={degeneracy_construction}\n")
    all_results = {}
    for pde, gen in GENS.items():
        print(f"\n{'='*60}\nPDE: {pde.upper()}\n{'='*60}")
        di, do, _ = gen(n_samples, nx, nt)
        ntr = int(0.8 * n_samples)
        tr_in, te_in = di[:ntr], di[ntr:]
        tr_out, te_out = do[:ntr], do[ntr:]
        pde_res = {}
        for mname, mtype, ctor in [
            ('FNO', 'fno', lambda: VanillaFNO2d(WIDTH, MODES, N_LAYERS)),
            ('EP-FNO', 'ep-fno', lambda: EP_FNO2d(WIDTH, MODES, N_LAYERS)),
            ('GENERIC-FNO', 'generic', lambda: GENERIC_FNO2d(
                nx, WIDTH_FUNC, MODES_FUNC, N_LAYERS_FUNC, MODES,
                l2_vargrad=l2_vargrad, use_residual=use_residual,
                degeneracy_construction=degeneracy_construction)),
        ]:
            print(f"\n--- {mname} ---")
            model = ctor()
            t0 = time.time()
            model = train_model(model, tr_in, tr_out, pde, mtype,
                                n_epochs=n_epochs, lr=1e-3, batch_size=BATCH,
                                device=device,
                                e_sup_mode=(e_sup_mode if mtype == 'generic' else 'none'),
                                min_diss_weight=(min_diss_weight if mtype == 'generic' else 0.0))
            r = evaluate_model(model, te_in, te_out, pde, mtype, device=device,
                               n_rollout=10, eval_batch=4)
            r['time'] = time.time() - t0
            r['params'] = count_params(model)
            pde_res[mname] = r
            print(f"  L2-1step={r['l2_single']:.6f} rollout={r['l2_rollout']:.6f} "
                  f"E-track={r['energy_track']:.6f} Mono={r['mono_violations']:.2f} "
                  f"({r['time']:.1f}s)")
            if mtype == 'generic':
                print(f"  dE/step={r['dE_mean']:.2e} dS/step={r['dS_mean']:.2e}")
                # --- checkpoint + scale-invariant channel diagnostic ---
                if save_dir:
                    import os
                    os.makedirs(save_dir, exist_ok=True)
                    tag = '' if min_diss_weight == 0.0 else f'_mep{min_diss_weight:g}'
                    if not degeneracy_construction:
                        tag += '_legacy'
                    if l2_vargrad:
                        tag += '_l2'
                    if (not use_residual) and (not degeneracy_construction):
                        tag += '_nores'
                    ckpt = os.path.join(save_dir, f'generic_fno_2d_{pde}_nx{nx}{tag}.pt')
                    torch.save({'state_dict': model.state_dict(), 'nx': nx,
                                'pde': pde, 'min_diss_weight': min_diss_weight,
                                'l2_vargrad': l2_vargrad, 'use_residual': use_residual,
                                'config': (WIDTH_FUNC, MODES_FUNC,
                                N_LAYERS_FUNC, MODES)}, ckpt)
                    print(f"  saved checkpoint -> {ckpt}")
                diag = channel_diagnostics(model, te_in[:, :1, :, :], n_batch=8,
                                           Y=te_out[:, :1, :, :])
                r['diagnostic'] = diag
                print(f"  [diag] rho_M={diag['rho_M']:.4f}  r_S={diag['r_S']:.2e}  "
                      f"r_E={diag['r_E']:.2e}  L_mag={diag['L_mag']:.2e}  "
                      f"M_mag={diag['M_mag']:.2e}")
                print(f"  [gauge-inv] r_mech={diag['r_mech']:+.4f}  "
                      f"pi_model={diag['pi_model']:+.4e}  "
                      f"pi_true={diag.get('pi_true', float('nan')):+.4e}")
            del model
            if device == 'cuda':
                torch.cuda.empty_cache()
        all_results[pde] = pde_res

    # summary
    lines = ["=" * 80, "BENCHMARK RESULTS (no E-supervision)",
             f"Date {time.strftime('%Y-%m-%d %H:%M:%S')}  device {device}",
             f"nx={nx} n_samples={n_samples} epochs={n_epochs}", "=" * 80]
    for pde, pr in all_results.items():
        lines.append(f"\n{pde.upper()}")
        lines.append(f"  {'Model':<13} {'Params':>9} {'L2-1step':>10} {'L2-roll':>10} "
                     f"{'E-track':>10} {'Mono':>8}")
        lines.append("  " + "-" * 64)
        for mn, r in pr.items():
            lines.append(f"  {mn:<13} {r['params']:>9,} {r['l2_single']:>10.6f} "
                         f"{r['l2_rollout']:>10.6f} {r['energy_track']:>10.6f} "
                         f"{r['mono_violations']:>8.2f}")
        if 'GENERIC-FNO' in pr:
            f_, g_ = pr['FNO'], pr['GENERIC-FNO']
            d = (g_['l2_rollout']-f_['l2_rollout'])/(f_['l2_rollout']+1e-10)*100
            lines.append(f"  -> GENERIC vs FNO L2-roll: {d:+.1f}%   "
                         f"dE/step={g_['dE_mean']:.1e} dS/step={g_['dS_mean']:.1e}")
    summary = "\n".join(lines)
    print("\n" + summary)

    # --- generator-channel diagnostic summary ---
    diag_lines = ["", "=" * 88,
                  "GENERATOR-CHANNEL DIAGNOSTIC",
                  "  gauge-dependent: rho_M, r_S (M-channel fraction / learned-entropy production)",
                  "  GAUGE-INVARIANT: r_mech, pi (dissipation of fixed Q=0.5||u||^2; pi_true=ground truth)",
                  "  reversible (advection) => all -> 0; dissipative (heat,burgers) => positive;",
                  "  wave is u-only (non-Markovian) so it dissipates observable Q (energy -> hidden velocity).",
                  "=" * 88,
                  f"  {'PDE':<9}{'rho_M':>9}{'r_S':>10}{'r_mech':>10}{'pi_model':>11}{'pi_true':>11}{'r_E':>11}",
                  "  " + "-" * 70]
    for pde, pr in all_results.items():
        if 'GENERIC-FNO' in pr and 'diagnostic' in pr['GENERIC-FNO']:
            d = pr['GENERIC-FNO']['diagnostic']
            diag_lines.append(f"  {pde:<9}{d['rho_M']:>9.4f}{d['r_S']:>10.2e}"
                              f"{d['r_mech']:>+10.4f}{d['pi_model']:>+11.3e}"
                              f"{d.get('pi_true', float('nan')):>+11.3e}{d['r_E']:>11.2e}")
    diag_lines += ["", "  LaTeX rows (PDE / rho_M / r_S / r_mech / pi_model / pi_true):"]
    for pde, pr in all_results.items():
        if 'GENERIC-FNO' in pr and 'diagnostic' in pr['GENERIC-FNO']:
            d = pr['GENERIC-FNO']['diagnostic']
            diag_lines.append(f"  {pde.capitalize():<8} & {d['rho_M']:.3f} & {d['r_S']:.1e} "
                              f"& {d['r_mech']:+.3f} & {d['pi_model']:+.2e} "
                              f"& {d.get('pi_true', float('nan')):+.2e} \\\\")
    diag_summary = "\n".join(diag_lines)
    print(diag_summary)
    summary = summary + "\n" + diag_summary

    if save_dir:
        import os
        os.makedirs(save_dir, exist_ok=True)
        ts = time.strftime('%Y%m%d_%H%M%S')
        with open(os.path.join(save_dir, f'benchmark_none_{ts}.pkl'), 'wb') as f:
            pickle.dump({'results': all_results, 'nx': nx, 'device': device}, f)
        with open(os.path.join(save_dir, f'benchmark_none_{ts}.txt'), 'w') as f:
            f.write(summary)
        print(f"\nSaved to {save_dir} (benchmark_none_{ts}.pkl/.txt + GENERIC checkpoints)")
    return all_results


# ============================================================================
# (b) INTERPRETABILITY PANEL — s_gauge: E conserved, S tracks dissipation
# ============================================================================

def run_interpretability_panel(save_dir=None, nx=64, n_train=120, nt=15,
                               n_epochs=120, n_traj=16):
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print("=" * 80)
    print(f"INTERPRETABILITY PANEL (s_gauge)  |  device={device}")
    print("Train GENERIC with S anchored to -0.5<u^2> (E free); then show that")
    print("along ground-truth trajectories E stays flat and S mirrors the decay.")
    print("=" * 80)
    WIDTH_FUNC, MODES_FUNC, N_LAYERS_FUNC, MODES = 24, 12, 3, 16
    PDES = ['heat', 'advection', 'burgers', 'wave']

    panel = {}   # panel[pde] = dict of (n_traj, T) arrays + summary
    for pde in PDES:
        print(f"\n{'='*60}\nPDE: {pde.upper()}\n{'='*60}")
        gen = GEN_BL[pde]
        tr_in, tr_out, _ = gen(n_train, nx, nt)
        te_in, _, _ = gen(n_traj, nx, nt)

        model = GENERIC_FNO2d(nx, WIDTH_FUNC, MODES_FUNC, N_LAYERS_FUNC, MODES)
        model = train_model(model, tr_in, tr_out, pde, 'generic',
                            n_epochs=n_epochs, lr=1e-3, batch_size=16,
                            device=device, e_sup_mode='s_gauge')

        model.eval()
        E_t, S_t, mechE_t = [], [], []
        with torch.no_grad():
            for i in range(te_in.shape[0]):
                u = te_in[i].to(device).unsqueeze(1)         # (T,1,H,W)
                E_t.append(model.E_net(u).cpu().numpy())
                S_t.append(model.S_net(u).cpu().numpy())
                mechE_t.append((0.5*(u**2).mean(dim=(-1,-2,-3))).cpu().numpy())
                del u
                if device == 'cuda':
                    torch.cuda.empty_cache()
        E_t = np.stack(E_t); S_t = np.stack(S_t); mechE_t = np.stack(mechE_t)  # (n_traj,T)

        # normalize each trajectory's E and S to [0,1]-ish for plotting overlay
        def _norm(a):
            mn = a.min(axis=1, keepdims=True); mx = a.max(axis=1, keepdims=True)
            return (a - mn) / (mx - mn + 1e-12)

        probes = probe_functionals(model, te_in, pde, device=device, n_probe=n_traj)
        diag = channel_diagnostics(model, te_in[:, :1, :, :], n_batch=8,
                                   Y=te_in[:, 1:2, :, :])
        panel[pde] = {'E_t': E_t, 'S_t': S_t, 'mechE_t': mechE_t,
                      'E_t_norm': _norm(E_t), 'S_t_norm': _norm(S_t),
                      'mechE_t_norm': _norm(mechE_t), 'probes': probes,
                      'diagnostic': diag}

        # textual panel
        E_mean = E_t.mean(0); S_mean = S_t.mean(0); m_mean = mechE_t.mean(0)
        E_cv = float(E_t.std(1).mean() / (np.abs(E_t).mean() + 1e-10))
        print(f"  [gauge-fixed diag] rho_M={diag['rho_M']:.4f} r_S={diag['r_S']:.3e} "
              f"r_E={diag['r_E']:.2e}")
        print(f"  mech-energy decay: {probes['mechE_decay']*100:5.1f}%  "
              f"| E_drift/mechE: {probes['E_drift_ratio']:.3f}  "
              f"| corr(S, mechE): {probes['corr_S_halfu2']:+.3f}")
        print(f"  E[u_t] (mean over traj): start={E_mean[0]:.4f} end={E_mean[-1]:.4f} "
              f"(CV={E_cv:.3f}  -> ~flat = conserved)")
        print(f"  S[u_t] (mean over traj): start={S_mean[0]:.4f} end={S_mean[-1]:.4f} "
              f"(monotone {'up' if S_mean[-1]>S_mean[0] else 'down'})")
        print(f"  mechE  (mean over traj): start={m_mean[0]:.4f} end={m_mean[-1]:.4f}")
        del model
        if device == 'cuda':
            torch.cuda.empty_cache()

    # gauge-fixed channel diagnostic summary (S anchored => physical gauge)
    print("\n" + "=" * 84)
    print("CHANNEL DIAGNOSTIC (s_gauge gauge for E,S; r_mech/pi are GAUGE-INVARIANT)")
    print("  reversible (advection) => everything ~ 0;  dissipative (heat,burgers) => positive")
    print("=" * 84)
    print(f"  {'PDE':<10}{'rho_M':>9}{'r_S':>10}{'r_mech':>10}{'pi_model':>11}"
          f"{'pi_true':>11}{'corr(S,mechE)':>15}")
    print("  " + "-" * 74)
    for pde in PDES:
        d = panel[pde]['diagnostic']; pb = panel[pde]['probes']
        print(f"  {pde:<10}{d['rho_M']:>9.4f}{d['r_S']:>10.2e}{d['r_mech']:>+10.4f}"
              f"{d['pi_model']:>+11.3e}{d.get('pi_true', float('nan')):>+11.3e}"
              f"{pb['corr_S_halfu2']:>+15.3f}")
    print("\n  LaTeX rows (PDE / r_mech / pi_model / pi_true / corr(S,mechE)):")
    for pde in PDES:
        d = panel[pde]['diagnostic']; pb = panel[pde]['probes']
        print(f"  {pde.capitalize():<10} & {d['r_mech']:+.3f} & {d['pi_model']:+.2e} "
              f"& {d.get('pi_true', float('nan')):+.2e} & {pb['corr_S_halfu2']:+.3f} \\\\")

    # optional matplotlib figure
    try:
        import matplotlib
        matplotlib.use('Agg')
        import matplotlib.pyplot as plt
        fig, axes = plt.subplots(1, len(PDES), figsize=(5*len(PDES), 4), squeeze=False)
        for j, pde in enumerate(PDES):
            ax = axes[0][j]
            d = panel[pde]
            T = d['E_t'].shape[1]
            tt = np.arange(T)
            for arr, lab, c in [('E_t_norm', 'learned E', 'C0'),
                                ('S_t_norm', 'learned S', 'C1'),
                                ('mechE_t_norm', '½⟨u²⟩ (mech E)', 'C2')]:
                m = d[arr].mean(0); s = d[arr].std(0)
                ax.plot(tt, m, color=c, label=lab)
                ax.fill_between(tt, m-s, m+s, color=c, alpha=0.15)
            ax.set_title(f"{pde}  (E_drift={d['probes']['E_drift_ratio']:.2f}, "
                         f"corr(S,mechE)={d['probes']['corr_S_halfu2']:+.2f})")
            ax.set_xlabel('rollout step'); ax.set_ylabel('normalized')
            if j == 0:
                ax.legend(fontsize=8)
        fig.tight_layout()
        if save_dir:
            import os
            os.makedirs(save_dir, exist_ok=True)
            ts = time.strftime('%Y%m%d_%H%M%S')
            png = os.path.join(save_dir, f'interpretability_panel_{ts}.png')
            fig.savefig(png, dpi=140)
            print(f"\nFigure saved: {png}")
    except Exception as ex:
        print(f"(matplotlib plot skipped: {ex})")

    if save_dir:
        import os
        os.makedirs(save_dir, exist_ok=True)
        ts = time.strftime('%Y%m%d_%H%M%S')
        with open(os.path.join(save_dir, f'interpretability_panel_{ts}.pkl'), 'wb') as f:
            pickle.dump({'panel': panel, 'nx': nx, 'device': device}, f)
        print(f"Panel data saved: {save_dir}/interpretability_panel_{ts}.pkl")
    return panel


def run_in_colab():
    import os
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        save_dir = '/content/drive/MyDrive/GENERIC_FNO_results'
    except ImportError:
        save_dir = './GENERIC_FNO_results'
    os.makedirs(save_dir, exist_ok=True)
    print(f"Results -> {save_dir}\n")

    print("\n" + "#"*80 + "\n# (a) Headline benchmark, no E-supervision\n" + "#"*80)
    bench = run_benchmark(save_dir=save_dir)   # nx=128 default

    print("\n" + "#"*80 + "\n# (b) Interpretability panel, s_gauge\n" + "#"*80)
    panel = run_interpretability_panel(save_dir=save_dir)  # nx=64 default

    return bench, panel


if __name__ == '__main__':
    run_in_colab()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Results -> /content/drive/MyDrive/GENERIC_FNO_results


################################################################################
# (a) Headline benchmark, no E-supervision
################################################################################
GENERIC-FNO 2D BENCHMARK (no E-supervision)  |  device=cuda
Config: nx=128 n_samples=150 nt=15 epochs=120 min_diss_weight=0.0 l2_vargrad=False use_residual=True degeneracy_construction=True


PDE: HEAT

--- FNO ---
  Epoch   1: data=0.004949
  Epoch  30: data=0.000010 rollout=0.000010
  Epoch  60: data=0.000011
  Epoch  90: data=0.000004 rollout=0.000002
  Epoch 120: rollout=0.000002 data=0.000002
  L2-1step=0.016674 rollout=0.095829 E-track=0.014939 Mono=3.33 (13.4s)

--- EP-FNO ---
  Epoch   1: data=0.022255 energy_penalty=0.000168
  Epoch  30: data=0.000042 energy_penalty=0.000000 rollout=0.000015
  